In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/studytrack_clustered.csv')
df.shape

(80000, 34)

In [2]:
cat_cols = ['gender', 'major', 'part_time_job', 'diet_quality', 'parental_education_level',
            'internet_quality', 'extracurricular_participation', 'dropout_risk',
            'study_environment', 'access_to_tutoring', 'family_income_range', 'learning_style']

df_model = df.copy()
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le

print("Encoded columns:", cat_cols)

Encoded columns: ['gender', 'major', 'part_time_job', 'diet_quality', 'parental_education_level', 'internet_quality', 'extracurricular_participation', 'dropout_risk', 'study_environment', 'access_to_tutoring', 'family_income_range', 'learning_style']


In [3]:
feature_cols = ['age', 'study_hours_per_day', 'social_media_hours', 'netflix_hours',
                 'part_time_job', 'attendance_percentage', 'sleep_hours', 'diet_quality',
                 'exercise_frequency', 'parental_education_level', 'internet_quality',
                 'mental_health_rating', 'extracurricular_participation', 'previous_gpa',
                 'stress_level', 'social_activity', 'screen_time', 'study_environment',
                 'access_to_tutoring', 'family_income_range', 'parental_support_level',
                 'motivation_level', 'exam_anxiety_score', 'learning_style',
                 'time_management_score', 'cluster']

X = df_model[feature_cols]
y = df_model['exam_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (64000, 26) Test shape: (16000, 26)


In [4]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

MAE:  3.247
RMSE: 4.186
R²:   0.871


In [7]:
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances.to_string(index=False))

                      feature  importance
                 previous_gpa    0.930305
        attendance_percentage    0.006386
        time_management_score    0.005500
                 stress_level    0.005402
                  sleep_hours    0.005209
         mental_health_rating    0.005144
          study_hours_per_day    0.005089
                  screen_time    0.004746
                netflix_hours    0.004501
           social_media_hours    0.004483
                          age    0.003146
       parental_support_level    0.002695
           exercise_frequency    0.002463
              social_activity    0.001970
             motivation_level    0.001961
            study_environment    0.001710
     parental_education_level    0.001700
               learning_style    0.001405
                 diet_quality    0.001005
             internet_quality    0.001000
          family_income_range    0.000985
                      cluster    0.000832
           exam_anxiety_score    0

In [8]:
habit_feature_cols = [c for c in feature_cols if c != 'previous_gpa']

X_habit = df_model[habit_feature_cols]
y_habit = df_model['exam_score']

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_habit, y_habit, test_size=0.2, random_state=42)

rf_habit_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_habit_model.fit(X_train_h, y_train_h)

y_pred_h = rf_habit_model.predict(X_test_h)
mae_h = mean_absolute_error(y_test_h, y_pred_h)
r2_h = r2_score(y_test_h, y_pred_h)

print(f"Habit-only model — MAE: {mae_h:.3f}, R²: {r2_h:.3f}")

Habit-only model — MAE: 8.655, R²: 0.181


In [9]:
importances_habit = pd.DataFrame({
    'feature': habit_feature_cols,
    'importance': rf_habit_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances_habit.to_string(index=False))

                      feature  importance
          study_hours_per_day    0.133051
                 stress_level    0.078841
                  sleep_hours    0.069722
        attendance_percentage    0.069160
             motivation_level    0.062446
        time_management_score    0.059560
         mental_health_rating    0.057815
                  screen_time    0.051925
                netflix_hours    0.049113
           social_media_hours    0.048497
            study_environment    0.043728
           exam_anxiety_score    0.039912
           exercise_frequency    0.036058
                          age    0.035529
       parental_support_level    0.031538
              social_activity    0.022794
           access_to_tutoring    0.021841
     parental_education_level    0.020288
               learning_style    0.016167
             internet_quality    0.011880
          family_income_range    0.011596
                 diet_quality    0.011168
                part_time_job    0

In [10]:
import joblib
import json

joblib.dump(rf_model, '../models/exam_score_predictor_full.pkl')
joblib.dump(rf_habit_model, '../models/exam_score_predictor_habits.pkl')

with open('../models/feature_columns.json', 'w') as f:
    json.dump({
        'full_model_features': feature_cols,
        'habit_model_features': habit_feature_cols
    }, f, indent=2)

# Save encoders so the backend can transform new user input the same way
joblib.dump(encoders, '../models/label_encoders.pkl')

print("All models and encoders saved.")

All models and encoders saved.


In [11]:
print(df_model['dropout_risk'].value_counts())
print(df_model['dropout_risk'].value_counts(normalize=True))

dropout_risk
0    78418
1     1582
Name: count, dtype: int64
dropout_risk
0    0.980225
1    0.019775
Name: proportion, dtype: float64


In [12]:
print(encoders['dropout_risk'].classes_)

['No' 'Yes']


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Exclude previous_gpa and exam_score to keep this focused on behavioral/lifestyle risk signals
# (a student's risk should be flagged from habits, not just restating their grades)
risk_feature_cols = [c for c in habit_feature_cols if c not in ['cluster']]

X_risk = df_model[risk_feature_cols]
y_risk = df_model['dropout_risk']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_risk, y_risk, test_size=0.2, random_state=42, stratify=y_risk
)

print("Train shape:", X_train_r.shape, "Test shape:", X_test_r.shape)
print("Train class balance:\n", y_train_r.value_counts(normalize=True))

Train shape: (64000, 24) Test shape: (16000, 24)
Train class balance:
 dropout_risk
0    0.980219
1    0.019781
Name: proportion, dtype: float64


In [14]:
rf_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train_r, y_train_r)
print("Classifier trained.")

Classifier trained.


In [15]:
y_pred_r = rf_classifier.predict(X_test_r)

print(classification_report(y_test_r, y_pred_r, target_names=['No Risk', 'At Risk']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_r, y_pred_r))

              precision    recall  f1-score   support

     No Risk       1.00      1.00      1.00     15684
     At Risk       1.00      1.00      1.00       316

    accuracy                           1.00     16000
   macro avg       1.00      1.00      1.00     16000
weighted avg       1.00      1.00      1.00     16000


Confusion Matrix:
[[15684     0]
 [    0   316]]


In [16]:
risk_importances = pd.DataFrame({
    'feature': risk_feature_cols,
    'importance': rf_classifier.feature_importances_
}).sort_values('importance', ascending=False)

print(risk_importances.to_string(index=False))

                      feature  importance
                 stress_level    0.596612
             motivation_level    0.245385
           exam_anxiety_score    0.116423
         mental_health_rating    0.012977
        attendance_percentage    0.003298
        time_management_score    0.002823
                  screen_time    0.002757
          study_hours_per_day    0.002734
                  sleep_hours    0.002555
           social_media_hours    0.002418
                netflix_hours    0.002142
                          age    0.001987
       parental_support_level    0.001227
           exercise_frequency    0.001045
              social_activity    0.000950
            study_environment    0.000907
     parental_education_level    0.000890
               learning_style    0.000597
                 diet_quality    0.000569
             internet_quality    0.000515
          family_income_range    0.000404
extracurricular_participation    0.000298
                part_time_job    0

In [17]:
for col in ['attendance_percentage', 'mental_health_rating', 'stress_level', 'exam_anxiety_score', 'time_management_score']:
    print(f"\n--- {col} vs dropout_risk ---")
    print(df.groupby('dropout_risk')[col].describe()[['mean', 'min', 'max']])


--- attendance_percentage vs dropout_risk ---
                   mean   min    max
dropout_risk                        
No            69.983339  40.0  100.0
Yes           69.201770  40.0   99.9

--- mental_health_rating vs dropout_risk ---
                  mean  min   max
dropout_risk                     
No            6.819957  1.0  10.0
Yes           6.018458  1.0  10.0

--- stress_level vs dropout_risk ---
                  mean  min   max
dropout_risk                     
No            4.934265  1.0  10.0
Yes           8.889381  8.1  10.0

--- exam_anxiety_score vs dropout_risk ---
                   mean   min   max
dropout_risk                       
No             8.478385   5.0  10.0
Yes           10.000000  10.0  10.0

--- time_management_score vs dropout_risk ---
                  mean  min   max
dropout_risk                     
No            5.500048  1.0  10.0
Yes           5.453729  1.0  10.0


In [18]:
# Instead of a black-box classifier reproducing a synthetic rule,
# we build a transparent, explainable risk score using the same real signals
# — this is more honest AND more useful for generating actionable recommendations.

def calculate_risk_factors(row):
    factors = []
    if row['stress_level'] >= 8:
        factors.append('High stress level')
    if row['exam_anxiety_score'] >= 8:
        factors.append('High exam anxiety')
    if row['motivation_level'] <= 4:
        factors.append('Low motivation')
    if row['attendance_percentage'] < 75:
        factors.append('Low attendance')
    if row['mental_health_rating'] <= 4:
        factors.append('Low mental health rating')
    if row['sleep_hours'] < 6:
        factors.append('Insufficient sleep')
    return factors

df['risk_factors'] = df.apply(calculate_risk_factors, axis=1)
df['risk_factor_count'] = df['risk_factors'].apply(len)

df['risk_factor_count'].value_counts().sort_index()

risk_factor_count
0     6411
1    18738
2    25610
3    21421
4     6900
5      874
6       46
Name: count, dtype: int64

In [19]:
def assign_risk_level(count):
    if count <= 1:
        return 'Low'
    elif count <= 3:
        return 'Moderate'
    else:
        return 'High'

df['risk_level'] = df['risk_factor_count'].apply(assign_risk_level)
print(df['risk_level'].value_counts())
print("\nAvg exam_score by risk_level:")
print(df.groupby('risk_level')['exam_score'].mean().sort_values(ascending=False))

risk_level
Moderate    47031
Low         25149
High         7820
Name: count, dtype: int64

Avg exam_score by risk_level:
risk_level
Low         92.304664
Moderate    88.165083
High        84.839642
Name: exam_score, dtype: float64


In [21]:
import joblib

# Retrain with a lighter configuration — still accurate, much smaller file size
rf_model_light = RandomForestRegressor(
    n_estimators=100, max_depth=10, min_samples_split=10, random_state=42, n_jobs=-1
)
rf_model_light.fit(X_train, y_train)

rf_habit_model_light = RandomForestRegressor(
    n_estimators=100, max_depth=10, min_samples_split=10, random_state=42, n_jobs=-1
)
rf_habit_model_light.fit(X_train_h, y_train_h)

# Re-check accuracy hasn't dropped meaningfully
from sklearn.metrics import r2_score, mean_absolute_error
print("Full model  -> MAE:", mean_absolute_error(y_test, rf_model_light.predict(X_test)),
      "R2:", r2_score(y_test, rf_model_light.predict(X_test)))
print("Habit model -> MAE:", mean_absolute_error(y_test_h, rf_habit_model_light.predict(X_test_h)),
      "R2:", r2_score(y_test_h, rf_habit_model_light.predict(X_test_h)))

# Save WITH compression this time — dramatically reduces file size
joblib.dump(rf_model_light, '../models/exam_score_predictor_full.pkl', compress=3)
joblib.dump(rf_habit_model_light, '../models/exam_score_predictor_habits.pkl', compress=3)

print("Lightweight compressed models saved.")

Full model  -> MAE: 3.2291054666568533 R2: 0.8714080484086448
Habit model -> MAE: 8.608116706906385 R2: 0.18657740605796735
Lightweight compressed models saved.
